In [13]:
# gradio_app_v3.py
import gradio as gr
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from imblearn.combine import SMOTETomek

# --- Load Trained Artifacts ---
model = joblib.load('/content/fetal_knn_model (1).pkl')
scaler = joblib.load('/content/fetal_scaler (1).pkl')
feature_names = joblib.load('/content/feature_names (2).pkl')
feature_ranges = joblib.load('/content/feature_ranges (1).pkl')

# --- Load Original Dataset to Build Correct Feature Ranges ---
df = pd.read_csv('fetal_health.csv')

feature_ranges = {}
for col in feature_names:
    min_val = df[col].min()
    max_val = df[col].max()
    step = 0.1
    feature_ranges[col] = (min_val, max_val, step)

# Save corrected ranges (optional)
joblib.dump(feature_ranges, "feature_ranges.pkl")


# --- SMOTE for Example Generation ---
X = df[feature_names]
y = df["fetal_health"]

sm = SMOTETomek(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X, y)

X_res = pd.DataFrame(X_resampled, columns=feature_names)
y_res = pd.Series(y_resampled)

# --- Preset Examples (Using Real Class Medians) ---
example_normal = X_res[y_res == 1].median().round(1).tolist()
example_suspect = X_res[y_res == 2].median().round(1).tolist()
example_pathological = X_res[y_res == 3].median().round(1).tolist()

# --- Label Mapping ---
labels = {
    1: "🟢 Normal",
    2: "🟡 Suspect",
    3: "🔴 Pathological"
}
colors = {1: "#2ECC71", 2: "#F39C12", 3: "#E74C3C"}


# --- Prediction + Plot ---
def predict_and_plot(*inputs):
    arr = np.array(inputs).reshape(1, -1)
    arr_scaled = scaler.transform(arr)

    pred_class = int(model.predict(arr_scaled)[0])
    pred_proba = model.predict_proba(arr_scaled)[0]

    fig, ax = plt.subplots(figsize=(5, 3))
    classes = [1, 2, 3]
    bars = ax.bar(
        [labels[c] for c in classes],
        pred_proba,
        color=[colors[c] for c in classes],
        edgecolor='white',
        linewidth=1.2
    )
    ax.set_ylim(0, 1)
    ax.set_ylabel("Probability")
    ax.set_title("Prediction Confidence")
    ax.grid(axis='y', linestyle='--', alpha=0.6)

    for bar, prob in zip(bars, pred_proba):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{prob:.1%}",
            ha='center',
            fontweight='bold'
        )

    plt.tight_layout()

    # Save record
    record = {feature_names[i]: inputs[i] for i in range(len(inputs))}
    record.update({
        "predicted_class": pred_class,
        "predicted_label": labels[pred_class],
        "prob_normal": pred_proba[0],
        "prob_suspect": pred_proba[1],
        "prob_pathological": pred_proba[2],
    })

    df_rec = pd.DataFrame([record])
    try:
        old = pd.read_csv("predictions_log.csv")
        df_rec = pd.concat([old, df_rec], ignore_index=True)
    except FileNotFoundError:
        pass

    df_rec.to_csv("predictions_log.csv", index=False)

    result_text = f"### 🎯 Prediction: **{labels[pred_class]}**\n"
    result_text += f"- **Confidence:** {pred_proba[pred_class - 1]:.1%}"

    return result_text, fig, "💾 Saved to predictions_log.csv"


# --- Gradio Theme ---
theme = gr.themes.Soft().set(
    button_primary_background_fill="*primary_500",
    button_primary_background_fill_hover="*primary_600"
)

# --- UI ---
with gr.Blocks(title="Fetal Health Classifier", theme=theme) as demo:
    gr.Markdown("# 🤰 **Fetal Health Classifier**")

    with gr.Row():
        with gr.Column(scale=2):
            gr.Markdown("### 📋 Input Features")
            sliders = []

            for feat in feature_names:
                min_val, max_val, step = feature_ranges[feat]
                slider = gr.Slider(
                    minimum=min_val,
                    maximum=max_val,
                    value=round((min_val + max_val) / 2, 1),
                    step=step,
                    label=feat.replace("_", " ").title(),
                )
                sliders.append(slider)

            gr.Markdown("### 🧪 Quick Examples")
            with gr.Row():
                b1 = gr.Button("Example 1")
                b2 = gr.Button("Example 2")
                b3 = gr.Button("Example 3")

            predict_btn = gr.Button("🔍 Predict & Save", variant="primary")
            status = gr.Textbox(label="", interactive=False)

        with gr.Column(scale=3):
            result = gr.Markdown()
            plot = gr.Plot()

    # Example loaders
    def load_example(vals):
        return {sliders[i]: vals[i] for i in range(len(sliders))}

    b1.click(lambda: load_example(example_normal), outputs=sliders)
    b2.click(lambda: load_example(example_suspect), outputs=sliders)
    b3.click(lambda: load_example(example_pathological), outputs=sliders)

    predict_btn.click(
        predict_and_plot,
        inputs=sliders,
        outputs=[result, plot, status]
    )

if __name__ == "__main__":
    demo.launch()

/tmp/ipython-input-4214870888.py:118: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Fetal Health Classifier", theme=theme) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://be0388c46b3d0f708a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
